# 🛍️ E-Commerce Sales Prediction

This project aims to predict the sales value for transactions in an online retail store using machine learning. We will use the UCI Online Retail Dataset.

## Project Steps:
1. **Data Preprocessing**: Cleaning and preparing the data.
2. **Exploratory Data Analysis (EDA)**: Understanding trends and patterns.
3. **Feature Engineering**: Creating features for our models.
4. **Model Training**: Training Linear Regression, Random Forest, and XGBoost.
5. **Evaluation**: Comparing model performance.
6. **Visualization**: Visualizing results and feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import sys
import os

# Add custom library path for disk space constraints if needed
sys.path.append(r'D:\pip_packages')

try:
    from xgboost import XGBRegressor
except ImportError:
    print("XGBoost not found. Please install it using: pip install xgboost")

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import joblib

# Create directories for outputs
os.makedirs('../outputs/plots', exist_ok=True)
print("Setup complete.")

## 🛠️ Step 1: Data Preprocessing

We will load the dataset and perform basic cleaning: removing nulls, duplicates, and invalid entries.

In [ ]:
print("Loading data...")
df = pd.read_excel('../data/Online_Retail.xlsx')

print("Cleaning data...")
# Drop rows with null CustomerID
df.dropna(subset=['CustomerID'], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove rows where Quantity <= 0 or UnitPrice <= 0
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Create Sales column
df['Sales'] = df['Quantity'] * df['UnitPrice']

# Feature Extraction from InvoiceDate
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Year'] = df['InvoiceDate'].dt.year
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['Hour'] = df['InvoiceDate'].dt.hour

# Label Encoding for Description
le = LabelEncoder()
df['Description'] = le.fit_transform(df['Description'].astype(str))

# Drop non-numeric and ID columns for modeling
df_model = df.drop(['InvoiceNo', 'CustomerID', 'Country', 'InvoiceDate'], axis=1)

print(f"Preprocessing complete. Remaining rows: {len(df_model)}")
df_model.head()

## 📊 Step 2: Exploratory Data Analysis (EDA)

Let's visualize the data distribution and trends.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df_model['Sales'], bins=50, kde=True)
plt.title('Sales Distribution (Top 95th Percentile)')
plt.xlim(0, df_model['Sales'].quantile(0.95))
plt.show()

plt.figure(figsize=(10, 6))
df.groupby('Month')['Sales'].sum().plot(kind='line', marker='o')
plt.title('Monthly Sales Trend')
plt.ylabel('Total Sales')
plt.show()

plt.figure(figsize=(12, 8))
sns.heatmap(df_model.corr(), annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

## ⚙️ Step 3: Train/Test Split

In [ ]:
X = df_model.drop('Sales', axis=1)
y = df_model['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Splitting complete.")

## 🤖 Step 4 & 5: Model Training and Evaluation

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({"Model": name, "R2": r2, "RMSE": rmse, "MAE": mae})

results_df = pd.DataFrame(results)
print(results_df)

## 📈 Step 6: Visualizations

Comparing the results visually.

In [ ]:
results_df.set_index('Model')[['R2', 'RMSE', 'MAE']].plot(kind='bar', subplots=True, layout=(1, 3), figsize=(15, 5))
plt.show()